# JOB 002 · Inspección cruda de la base SEPS + diagnóstico de morosidad

**Objetivo:** no asumir nada sobre la estructura del proyecto.  
Este notebook empieza por los archivos Parquet reales y responde, en este orden:

1. ¿Dónde están los Parquet?
2. ¿Qué archivos/particiones existen?
3. ¿Qué **columnas y tipos** trae realmente la base?
4. ¿Hay diferencias de esquema entre particiones?
5. ¿Qué entidades existen?
6. ¿Qué cuentas y descripciones existen realmente?
7. ¿Cómo aparece CPN?
8. ¿Existen las cuentas `14`, `1499` y las cuentas NPL?
9. ¿Por qué `morosidad` está quedando vacía?
10. ¿Qué fórmula puede reconstruirse con lo que realmente existe?

**Solo lectura.** No modifica Parquet, dashboard, Atlas ni configuración.

> **Corrección del JOB 001:** se corrigió el `BinderException` de DuckDB en consultas agrupadas (`GROUP BY account` + `ORDER BY length(account)`).  
> Este JOB 002 usa `GROUP BY account, n_digitos` y ordena por los alias ya agrupados. También se corrigió la misma construcción en la búsqueda de cuentas NPL.

In [ ]:
# 0 · ENTORNO Y LOCALIZACIÓN DEL PROYECTO
from pathlib import Path
import os, sys, json, re, math, statistics
import pandas as pd
import numpy as np
from IPython.display import display

# Montar Drive si estamos en Colab.
try:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

# Puedes cambiarlo manualmente si tu carpeta tiene otro nombre.
PROJECT_ROOT = Path('/content/drive/MyDrive/seps-segmento1-risk')

# Si no existe, intentar localizar automáticamente data/parquet/eeff.
if not PROJECT_ROOT.exists():
    candidates = []
    base = Path('/content/drive/MyDrive')
    if base.exists():
        for p in base.glob('**/data/parquet/eeff'):
            candidates.append(p.parents[2])
    if len(candidates) == 1:
        PROJECT_ROOT = candidates[0]
    elif len(candidates) > 1:
        print('Se encontraron varios proyectos:')
        for i,p in enumerate(candidates):
            print(i, p)
        raise RuntimeError('Define PROJECT_ROOT manualmente arriba.')
    else:
        raise FileNotFoundError('No encuentro el proyecto ni data/parquet/eeff.')

PARQUET_ROOT = PROJECT_ROOT / 'data/parquet/eeff'
OUTPUT_DIR = PROJECT_ROOT / 'notebooks/jobs/output/002_inspeccion_cruda'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT :', PROJECT_ROOT)
print('PARQUET_ROOT :', PARQUET_ROOT)
print('OUTPUT_DIR   :', OUTPUT_DIR)

In [ ]:
# 1 · DEPENDENCIAS MÍNIMAS (sin importar sepsrisk)
import importlib, subprocess, sys

for pkg in ['duckdb','pyarrow']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        print(f'Instalando {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import duckdb
import pyarrow as pa
import pyarrow.parquet as pq
print('duckdb:', duckdb.__version__)
print('pyarrow:', pa.__version__)

In [ ]:
# 2 · INVENTARIO REAL DE PARQUET
files = sorted(PARQUET_ROOT.glob('year=*/month=*/eeff.parquet'))

print(f'Archivos Parquet encontrados: {len(files)}')

inventory = []
for p in files:
    m_year = re.search(r'year=(\d{4})', str(p))
    m_month = re.search(r'month=(\d{2})', str(p))
    inventory.append({
        'year': int(m_year.group(1)) if m_year else None,
        'month': int(m_month.group(1)) if m_month else None,
        'path': str(p),
        'size_mb': p.stat().st_size / 1024**2,
    })

inventory_df = pd.DataFrame(inventory).sort_values(['year','month']) if inventory else pd.DataFrame()
display(inventory_df)

if not files:
    legacy = PROJECT_ROOT / 'data/operational/seps_segmento1.sqlite'
    print('No hay Parquet.')
    print('SQLite legacy existe:', legacy.exists(), legacy)
    raise RuntimeError('No hay Parquet que inspeccionar. Si esperabas datos, la migración/ingesta no llegó a escribirlos.')

print('\nPeriodo físico:', inventory_df[['year','month']].iloc[0].to_dict(), '→', inventory_df[['year','month']].iloc[-1].to_dict())
print(f'Tamaño total: {inventory_df.size_mb.sum():,.2f} MB')

inventory_df.to_csv(OUTPUT_DIR/'inventario_parquet.csv', index=False)

In [ ]:
# 3 · ESQUEMA CRUDO DE CADA PARTICIÓN
schemas = []
schema_signatures = {}

for p in files:
    sch = pq.read_schema(p)
    sig = tuple((f.name, str(f.type)) for f in sch)
    schema_signatures.setdefault(sig, []).append(str(p))
    for f in sch:
        schemas.append({
            'file': str(p),
            'column': f.name,
            'type': str(f.type),
            'nullable': f.nullable,
        })

schema_df = pd.DataFrame(schemas)

print('=== COLUMNAS REALES DE LA BASE ===')
first_schema = pq.read_schema(files[0])
columns_df = pd.DataFrame([
    {'orden': i+1, 'campo': f.name, 'tipo': str(f.type), 'nullable': f.nullable}
    for i,f in enumerate(first_schema)
])
display(columns_df)

print(f'\nFirmas de esquema distintas entre particiones: {len(schema_signatures)}')
for i,(sig, paths) in enumerate(schema_signatures.items(), start=1):
    print(f'\nESQUEMA {i}: {len(paths)} archivos')
    print(pd.DataFrame(sig, columns=['campo','tipo']).to_string(index=False))
    if len(paths) <= 5:
        print('\n'.join(paths))
    else:
        print('Ejemplos:', *paths[:3], sep='\n  ')

columns_df.to_csv(OUTPUT_DIR/'campos_base.csv', index=False)
schema_df.to_csv(OUTPUT_DIR/'esquemas_por_particion.csv', index=False)

In [ ]:
# 4 · ABRIR LOS PARQUET DIRECTAMENTE CON DUCKDB
glob = (PARQUET_ROOT / 'year=*/month=*/eeff.parquet').as_posix()
con = duckdb.connect(':memory:')

con.execute(f"""
CREATE VIEW raw_eeff AS
SELECT *
FROM read_parquet('{glob}', hive_partitioning=true, union_by_name=true)
""")

print('=== DESCRIBE raw_eeff ===')
describe = con.execute('DESCRIBE raw_eeff').df()
display(describe)

print('\n=== 20 FILAS CRUDAS ===')
sample = con.execute('SELECT * FROM raw_eeff LIMIT 20').df()
display(sample)

describe.to_csv(OUTPUT_DIR/'describe_raw_eeff.csv', index=False)
sample.to_csv(OUTPUT_DIR/'muestra_20_filas.csv', index=False)

In [ ]:
# 5 · PERFIL DE CADA CAMPO
cols = [r[0] for r in con.execute('DESCRIBE raw_eeff').fetchall()]

profile_rows = []
n_total = con.execute('SELECT count(*) FROM raw_eeff').fetchone()[0]

for c in cols:
    q = f'''
    SELECT
      count(*) AS n,
      count("{c}") AS non_null,
      count(*) - count("{c}") AS nulls,
      approx_count_distinct("{c}") AS approx_distinct
    FROM raw_eeff
    '''
    r = con.execute(q).fetchone()
    profile_rows.append({
        'campo': c,
        'filas': r[0],
        'no_nulos': r[1],
        'nulos': r[2],
        'pct_nulos': (r[2]/r[0]*100) if r[0] else np.nan,
        'distinct_aprox': r[3],
    })

profile = pd.DataFrame(profile_rows)
print(f'Filas totales raw_eeff: {n_total:,}')
display(profile)
profile.to_csv(OUTPUT_DIR/'perfil_campos.csv', index=False)

In [ ]:
# 6 · ENTIDADES REALES DISPONIBLES
required = {'ruc','entity_name'}
missing = required - set(cols)
if missing:
    raise RuntimeError(f'Faltan columnas esperadas para entidad: {missing}')

entities = con.execute('''
SELECT
  CAST(ruc AS VARCHAR) AS ruc,
  any_value(CAST(entity_name AS VARCHAR)) AS entity_name,
  min(CAST(cutoff_date AS VARCHAR)) AS first_date,
  max(CAST(cutoff_date AS VARCHAR)) AS last_date,
  count(*) AS rows,
  count(DISTINCT CAST(cutoff_date AS VARCHAR)) AS months
FROM raw_eeff
GROUP BY 1
ORDER BY entity_name
''').df()

print('Entidades:', len(entities))
display(entities)
entities.to_csv(OUTPUT_DIR/'entidades.csv', index=False)

# Resolver CPN por texto, mostrando todos los matches.
cpn_matches = entities[
    entities.entity_name.str.upper().str.contains('POLICIA NACIONAL', na=False, regex=False)
].copy()

print('\n=== MATCHES CPN ===')
display(cpn_matches)

if len(cpn_matches) != 1:
    raise RuntimeError(f'Esperaba un único match para POLICIA NACIONAL y encontré {len(cpn_matches)}.')

CPN_RUC = str(cpn_matches.iloc[0].ruc)
CPN_NAME = str(cpn_matches.iloc[0].entity_name)
print('CPN_RUC :', CPN_RUC)
print('CPN_NAME:', CPN_NAME)

In [ ]:
# 7 · CATÁLOGO REAL DE CUENTAS: AQUÍ NO ASUMIMOS NADA
required = {'account','account_description','balance','cutoff_date'}
missing = required - set(cols)
if missing:
    raise RuntimeError(f'Faltan columnas contables esperadas: {missing}')

accounts = con.execute('''
SELECT
  CAST(account AS VARCHAR) AS account,
  any_value(CAST(account_description AS VARCHAR)) AS account_description,
  length(CAST(account AS VARCHAR)) AS n_digitos,
  min(CAST(cutoff_date AS VARCHAR)) AS first_date,
  max(CAST(cutoff_date AS VARCHAR)) AS last_date,
  count(*) AS rows,
  count(DISTINCT CAST(cutoff_date AS VARCHAR)) AS months,
  count(DISTINCT CAST(ruc AS VARCHAR)) AS entities
FROM raw_eeff
GROUP BY 1,3
ORDER BY n_digitos, account
''').df()

print('Cuentas únicas globales:', len(accounts))
print('\n=== DISTRIBUCIÓN POR NÚMERO DE DÍGITOS ===')
display(accounts.groupby('n_digitos').agg(
    cuentas=('account','count'),
    first_date=('first_date','min'),
    last_date=('last_date','max')
).reset_index())

print('\n=== PRIMERAS 300 CUENTAS REALES ===')
display(accounts.head(300))

accounts.to_csv(OUTPUT_DIR/'catalogo_cuentas_global.csv', index=False)

In [ ]:
# 8 · TODAS LAS CUENTAS DE CPN EN SU ÚLTIMO CORTE
last_date = con.execute(
    'SELECT max(CAST(cutoff_date AS VARCHAR)) FROM raw_eeff WHERE CAST(ruc AS VARCHAR)=?',
    [CPN_RUC]
).fetchone()[0]

cpn_last = con.execute('''
SELECT
  CAST(account AS VARCHAR) AS account,
  CAST(account_description AS VARCHAR) AS account_description,
  CAST(balance AS DOUBLE) AS balance,
  length(CAST(account AS VARCHAR)) AS n_digitos
FROM raw_eeff
WHERE CAST(ruc AS VARCHAR)=?
  AND CAST(cutoff_date AS VARCHAR)=?
ORDER BY n_digitos, account
''', [CPN_RUC, last_date]).df()

print('Último corte CPN:', last_date)
print('Filas/cuentas en último corte:', len(cpn_last))
display(cpn_last)

cpn_last.to_csv(OUTPUT_DIR/'cpn_todas_las_cuentas_ultimo_corte.csv', index=False)

In [ ]:
# 9 · CUENTAS 14* DE CPN — SIN FILTRO DE FEATURES
cpn_14 = con.execute('''
SELECT
  CAST(cutoff_date AS VARCHAR) AS cutoff_date,
  CAST(account AS VARCHAR) AS account,
  CAST(account_description AS VARCHAR) AS account_description,
  CAST(balance AS DOUBLE) AS balance,
  length(CAST(account AS VARCHAR)) AS n_digitos
FROM raw_eeff
WHERE CAST(ruc AS VARCHAR)=?
  AND CAST(account AS VARCHAR) LIKE '14%'
ORDER BY cutoff_date, length(account), account
''', [CPN_RUC]).df()

print('Filas 14* CPN:', len(cpn_14))
print('Cuentas 14* únicas:', cpn_14.account.nunique())

print('\n=== CUENTAS 14* ÚNICAS ===')
catalog14 = (cpn_14[['account','account_description','n_digitos']]
             .drop_duplicates()
             .sort_values(['n_digitos','account']))
display(catalog14)

print('\n=== 14* ÚLTIMO CORTE ===')
display(cpn_14[cpn_14.cutoff_date == last_date])

catalog14.to_csv(OUTPUT_DIR/'cpn_catalogo_14.csv', index=False)
cpn_14.to_csv(OUTPUT_DIR/'cpn_serie_cuentas_14.csv', index=False)

In [ ]:
# 10 · BUSCAR CONCEPTOS POR DESCRIPCIÓN REAL
patterns = {
    'cartera': 'CARTERA',
    'vencida': 'VENCID',
    'no_devenga': 'NO DEVENG',
    'improductiva': 'IMPRODUCT',
    'provision': 'PROVIS',
    'reestructurada': 'REESTRUCT',
    'refinanciada': 'REFINANC',
}

concept_rows = []
for label, patt in patterns.items():
    df = con.execute('''
    SELECT DISTINCT
      CAST(account AS VARCHAR) AS account,
      CAST(account_description AS VARCHAR) AS account_description,
      length(CAST(account AS VARCHAR)) AS n_digitos
    FROM raw_eeff
    WHERE CAST(ruc AS VARCHAR)=?
      AND upper(CAST(account_description AS VARCHAR)) LIKE ?
    ORDER BY n_digitos, account
    ''', [CPN_RUC, f'%{patt}%']).df()
    df.insert(0, 'concepto', label)
    concept_rows.append(df)

concepts = pd.concat(concept_rows, ignore_index=True) if concept_rows else pd.DataFrame()
display(concepts)
concepts.to_csv(OUTPUT_DIR/'cpn_cuentas_por_concepto.csv', index=False)

In [ ]:
# 11 · EXISTENCIA DE CUENTAS EXACTAS QUE USA EL MODELO ACTUAL
# Fórmula actual del proyecto:
# loan_portfolio_net = cuenta 14
# loan_provisions    = cuenta 1499
# loan_portfolio_gross = loan_portfolio_net - loan_provisions

exact = ['14','1499']
exact_status = []

for acc in exact:
    df = con.execute('''
    SELECT
      count(*) AS rows,
      count(DISTINCT CAST(cutoff_date AS VARCHAR)) AS months,
      min(CAST(cutoff_date AS VARCHAR)) AS first_date,
      max(CAST(cutoff_date AS VARCHAR)) AS last_date,
      any_value(CAST(account_description AS VARCHAR)) AS description
    FROM raw_eeff
    WHERE CAST(ruc AS VARCHAR)=?
      AND CAST(account AS VARCHAR)=?
    ''', [CPN_RUC, acc]).df()
    row = df.iloc[0].to_dict()
    row['account'] = acc
    exact_status.append(row)

exact_status = pd.DataFrame(exact_status)[
    ['account','description','rows','months','first_date','last_date']
]
print('=== ¿EXISTEN 14 Y 1499 EXACTAMENTE? ===')
display(exact_status)
exact_status.to_csv(OUTPUT_DIR/'cpn_existencia_14_1499.csv', index=False)

In [ ]:
# 12 · SERIES EXACTAS 14 Y 1499
series_exact = con.execute('''
SELECT
  CAST(cutoff_date AS VARCHAR) AS cutoff_date,
  max(CASE WHEN CAST(account AS VARCHAR)='14' THEN CAST(balance AS DOUBLE) END) AS acc_14,
  max(CASE WHEN CAST(account AS VARCHAR)='1499' THEN CAST(balance AS DOUBLE) END) AS acc_1499
FROM raw_eeff
WHERE CAST(ruc AS VARCHAR)=?
GROUP BY 1
ORDER BY 1
''', [CPN_RUC]).df()

series_exact['gross_if_formula_current'] = series_exact['acc_14'] - series_exact['acc_1499']

display(series_exact.tail(36))
series_exact.to_csv(OUTPUT_DIR/'cpn_series_14_1499.csv', index=False)

In [ ]:
# 13 · TOTALES 14* POR NIVEL CONTABLE: DIAGNÓSTICO DE JERARQUÍA
# Esto NO afirma que se deban sumar niveles. Sirve para ver qué nivel trae los totales.
totals_by_level = con.execute('''
SELECT
  CAST(cutoff_date AS VARCHAR) AS cutoff_date,
  length(CAST(account AS VARCHAR)) AS n_digitos,
  count(*) AS n_cuentas,
  sum(CAST(balance AS DOUBLE)) AS suma_saldos
FROM raw_eeff
WHERE CAST(ruc AS VARCHAR)=?
  AND CAST(account AS VARCHAR) LIKE '14%'
GROUP BY 1,2
ORDER BY 1,2
''', [CPN_RUC]).df()

pivot_levels = totals_by_level.pivot(
    index='cutoff_date', columns='n_digitos', values='suma_saldos'
).sort_index()

print('=== SUMA DE 14* POR NIVEL ===')
display(pivot_levels.tail(36))

totals_by_level.to_csv(OUTPUT_DIR/'cpn_totales_14_por_nivel.csv', index=False)

In [ ]:
# 14 · NPL: BUSCAR DIRECTAMENTE CÓMO ESTÁ REPRESENTADA EN LA BASE
# Primero por descripciones; después mostraremos las cuentas candidatas.
npl_desc = con.execute('''
SELECT
  CAST(account AS VARCHAR) AS account,
  any_value(CAST(account_description AS VARCHAR)) AS account_description,
  length(CAST(account AS VARCHAR)) AS n_digitos,
  count(DISTINCT CAST(cutoff_date AS VARCHAR)) AS months
FROM raw_eeff
WHERE CAST(ruc AS VARCHAR)=?
  AND (
       upper(CAST(account_description AS VARCHAR)) LIKE '%NO DEVENG%'
    OR upper(CAST(account_description AS VARCHAR)) LIKE '%VENCID%'
    OR upper(CAST(account_description AS VARCHAR)) LIKE '%IMPRODUCT%'
  )
GROUP BY 1,3
ORDER BY n_digitos, account
''', [CPN_RUC]).df()

print('=== CANDIDATAS A CARTERA IMPRODUCTIVA POR DESCRIPCIÓN ===')
display(npl_desc)
npl_desc.to_csv(OUTPUT_DIR/'cpn_cuentas_npl_por_descripcion.csv', index=False)

In [ ]:
# 15 · DIAGNÓSTICO AUTOMÁTICO
messages = []

has14 = bool((exact_status.account == '14').any() and exact_status.loc[exact_status.account=='14','rows'].iloc[0] > 0)
has1499 = bool((exact_status.account == '1499').any() and exact_status.loc[exact_status.account=='1499','rows'].iloc[0] > 0)

if not has14:
    messages.append('CRÍTICO: la cuenta exacta 14 NO existe para CPN. La fórmula actual de loan_portfolio_net no puede funcionar.')
else:
    messages.append('OK: la cuenta exacta 14 existe para CPN.')

if not has1499:
    messages.append('CRÍTICO: la cuenta exacta 1499 NO existe para CPN. La fórmula actual de provisiones no puede funcionar.')
else:
    messages.append('OK: la cuenta exacta 1499 existe para CPN.')

if len(npl_desc) == 0:
    messages.append('CRÍTICO: no encontré cuentas con descripciones NO DEVENGA / VENCIDA / IMPRODUCTIVA para CPN.')
else:
    messages.append(f'INFO: encontré {len(npl_desc)} cuentas candidatas NPL por descripción.')

if series_exact['acc_14'].isna().all():
    messages.append('CRÍTICO: acc_14 es NaN en toda la serie.')
elif series_exact['acc_14'].isna().any():
    messages.append(f'ATENCIÓN: acc_14 falta en {series_exact.acc_14.isna().sum()} meses.')

if series_exact['acc_1499'].isna().all():
    messages.append('CRÍTICO: acc_1499 es NaN en toda la serie.')
elif series_exact['acc_1499'].isna().any():
    messages.append(f'ATENCIÓN: acc_1499 falta en {series_exact.acc_1499.isna().sum()} meses.')

print('\n'.join('• '+m for m in messages))

report = {
    'project_root': str(PROJECT_ROOT),
    'parquet_files': len(files),
    'raw_rows': int(n_total),
    'columns': cols,
    'entities': int(len(entities)),
    'cpn_ruc': CPN_RUC,
    'cpn_name': CPN_NAME,
    'last_date': str(last_date),
    'unique_accounts_global': int(len(accounts)),
    'unique_accounts_14_cpn': int(cpn_14.account.nunique()),
    'has_exact_14': has14,
    'has_exact_1499': has1499,
    'npl_candidates_by_description': int(len(npl_desc)),
    'messages': messages,
}

with open(OUTPUT_DIR/'diagnostico.json','w',encoding='utf-8') as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print('\nDiagnóstico guardado en:', OUTPUT_DIR/'diagnostico.json')

## Qué pasarme después de ejecutar

Con **Ejecutar todo**, pásame preferentemente:

- `notebooks/jobs/output/002_inspeccion_cruda/diagnostico.json`
- `cpn_existencia_14_1499.csv`
- `cpn_catalogo_14.csv`
- `cpn_cuentas_npl_por_descripcion.csv`

o simplemente capturas/salida de las secciones:

**3 · ESQUEMA**, **7 · CATÁLOGO REAL**, **11 · EXISTENCIA 14/1499** y **15 · DIAGNÓSTICO AUTOMÁTICO**.

Con eso corregimos la fórmula de morosidad usando **lo que realmente trae SEPS**, no supuestos.